In [0]:
%run ../../config/utils

In [0]:
%run ../../lib/email_sender_databricks

In [0]:
from datetime import date, timedelta, timezone, datetime
import re
from pyspark.sql import functions as f

In [0]:
dbutils.widgets.text("target_emails", "", "Comma-separated list of email addresses to send to")
dbutils.widgets.text("subject", "", "Subject of the email")
dbutils.widgets.text("body", "", "Body of the email")

In [0]:
def recent_saturday_str(today: date | None = None) -> str:
    if today is None:
        today = date.today()
    offset = (today.weekday() + 2) % 7
    return (today - timedelta(days=offset))

last_saturday       = recent_saturday_str()
last_saturday_str   = last_saturday.strftime('%Y-%m-%d')
RECENT_SAT          = last_saturday.strftime('%Y%m%d')

In [0]:
file_patterns = [
    's3://memberanalytics-data-in-prod/Master_Data/kantar_itemmaster_hierarchy_data_SAP_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/Master_Data/Itemmaster_brand_data_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/Transaction_Detail/1010_Transaction_Detail_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/Transaction_Payment/1010_Transaction_Payment_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/Membership/DataLogix_customer_data_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/Membership/BCG_extended_mbrshp_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/Other/MEMBER_TRACTS.csv',
    's3://memberanalytics-data-in-prod/Other/Tract_Zip_Code_BJS_Costco_Sams_Walmart_Distances.csv',
    's3://memberanalytics-data-in-prod/Membership/BCG_Member_Hist_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/Master_Data/kantar_storemaster_hierarchy_data_SAP_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/Email_Events/PE_email_event_%(curr_date)s.csv',
    's3://memberanalytics-data-in-prod/ATC/BJs_CRM_*',
    's3://memberanalytics-data-in-prod/ATC/bjs_loyaltynumbers_usercodes_10182019.csv',
    's3://memberanalytics-data-in-prod/control_files/tab_01_redshift_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/control_files/tab_02_redshift_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/control_files/tab_03_redshift_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/control_files/tab_04_redshift_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/control_files/tab_05_redshift_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/control_files/tab_06_redshift_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/control_files/tab_07_redshift_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/control_files/tab_08_redshift_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/control_files/tab_09_redshift_%(curr_date)s.txt',
    's3://memberanalytics-data-in-prod/Membership/PE_Member_Award_Certificate/PE_Member_Award_Certificate_%(curr_date)s.csv',
    's3://memberanalytics-data-in-prod/ATC/BJs_loyaltyNumber_userCode_mapping_%(curr_date)s.csv'
]

In [0]:
def resolve_curr_date(p: str, yyyymmdd: str) -> str:
    return p.replace('%(curr_date)s', yyyymmdd)

resolved_patterns = [resolve_curr_date(p, RECENT_SAT) for p in file_patterns]

In [0]:
def _top_level_prefix_path(s3_url: str) -> str:
    """
    Return the top-level prefix path for recursive listing, e.g.:
      s3://bucket/Master_Data/...
    """
    assert s3_url.startswith('s3://'), f'Unexpected scheme in {s3_url}'
    parts = s3_url.replace('s3://', '').split('/')
    if len(parts) < 2:
        # just the bucket; list it
        return f's3://{parts[0]}/'
    return f's3://{parts[0]}/{parts[1]}/'

def _pattern_to_regex(s3_url_pattern: str) -> str:
    """
    Convert an S3-style '*' wildcard pattern into a full regex over the absolute path.
    Only '*' is treated as a wildcard; other regex special chars are escaped.
    """
    esc = re.escape(s3_url_pattern)
    esc = esc.replace(r'\*', '.*')
    return f'^{esc}$'

def _list_paths_recursive(base_prefix: str):
    """
    Use Spark binaryFile to list all objects recursively under base_prefix.
    Returns a DataFrame with column 'path'.
    """
    df = (spark.read.format('binaryFile')
          .option('recursiveFileLookup', 'true')
          .load(base_prefix)
          .select(f.col('path')))
    return (df.filter(~f.col('path').rlike(r'/_temporary/|/_SUCCESS($|/)'))
              .filter(~f.col('path').endswith('.crc')))

def check_file_databricks(resolved_pattern: str, original_pattern: str) -> str:
    """
    Reproduces the original notebook's checks with Databricks-native listing.
    - If the top-level prefix has no objects -> "(missing)"
    - If the pattern had no %(curr_date)s and no '*' -> "(present)" if exact file exists, else "(not found)"
    - If the pattern contained '*' -> "(present)" if any object matches the wildcard regex, else "(not found)"
    - If the pattern had %(curr_date)s (i.e., is a dated pattern) -> "(present)" only if the dated file exists;
      otherwise "(no recent Saturday file)"
    """
    base = _top_level_prefix_path(resolved_pattern)
    df_all = _list_paths_recursive(base)

    if df_all.select('path').limit(1).count() == 0:
        return f"{resolved_pattern} (missing)"

    # Wildcard?
    has_wildcard = '*' in original_pattern
    is_dated = '%(curr_date)s' in original_pattern

    if has_wildcard:
        rx = _pattern_to_regex(resolved_pattern)
        cnt = df_all.filter(f.col('path').rlike(rx)).limit(1).count()
        return f"{resolved_pattern} (present)" if cnt > 0 else f"{resolved_pattern} (not found)"

    # No wildcard: exact match by full path (after resolving date if any)
    cnt = (df_all.filter(f.col('path') == resolved_pattern).limit(1).count())
    if cnt > 0:
        return f"{resolved_pattern} (present)"

    # Not found: distinguish dated vs. undated
    if is_dated:
        return f"{resolved_pattern} (no recent Saturday file)"
    else:
        return f"{resolved_pattern} (not found)"

In [0]:
output_lines = [f"Recent Saturday: {RECENT_SAT}"]

results = [check_file_databricks(resolved, original) for resolved, original in zip(resolved_patterns, file_patterns)]

output_lines.append("File Check Results:")
output_lines.extend(results)
output_report_str = "\n".join(output_lines)

print(output_report_str)

In [0]:
# Store report for notify_started to use
dbutils.jobs.taskValues.set(key="file_check_report", value=output_report_str)
print(output_report_str)